In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset
from trl import GRPOTrainer, GRPOConfig
from peft import LoraConfig, PeftModel
import json
import wandb

wandb.init(project='mini-llm', name='GRPO')

## Загрузка моделей

- **Policy** — SFT checkpoint. GRPOTrainer обернёт его в LoRA (peft_config).
  Базовые веса заморожены → reference. Адаптеры обучаются → policy.
- **Reward Model** — загружаем сами как `AutoModelForSequenceClassification(num_labels=1)`.
  Передадим объект модели в `reward_funcs`, а его токенизатор — в `reward_processing_classes`.

In [ ]:
from transformers import AutoModelForSequenceClassification

sft_path = "./checkpoints/sft/sft-seed42/final"
rm_path = "./checkpoints/rm/final"

model = AutoModelForCausalLM.from_pretrained(sft_path, dtype=torch.bfloat16)

tokenizer = AutoTokenizer.from_pretrained(sft_path)
tokenizer.padding_side = "left"
tokenizer.add_special_tokens({"pad_token": "<|pad|>"})
model.resize_token_embeddings(len(tokenizer))

rm_model = AutoModelForSequenceClassification.from_pretrained(
    rm_path, num_labels=1, dtype=torch.bfloat16
)
rm_tokenizer = AutoTokenizer.from_pretrained(rm_path)

print(f"Policy: {model.config._name_or_path}")
print(f"RM loaded from {rm_path}")

## Датасет промптов

Берём те же промпты что и для DPO (из dpo_pairs.json) — для честного сравнения.
Но пары (chosen/rejected) нам не нужны — GRPO сгенерирует ответы сам.

Формат: `{"prompt": [{"role": "user", "content": "..."}]}` — GRPOTrainer
сам применит chat template.

In [ ]:
random_state = 42

with open("dpo_pairs.json") as f:
    data = json.load(f)

rows = [{"prompt": [{"role": "user", "content": p["prompt"]}]} for p in data["pairs"]]
dataset = Dataset.from_list(rows).shuffle(seed=random_state)

eval_size = 200
val_dataset = dataset.select(range(eval_size))
train_dataset = dataset.select(range(eval_size, len(dataset)))

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")
print(f"Пример промпта: {train_dataset[0]['prompt'][0]['content'][:100]}")

## Конфиг и обучение

Ключевые параметры GRPO:
- `num_generations=4` — сколько ответов генерировать на каждый промпт
- `max_completion_length=256` — макс. длина генерации
- `temperature=0.9` — температура генерации (выше → разнообразнее ответы в группе)
- `scale_rewards="group"` — нормализация наград внутри группы (mean=0, std=1)

LoRA — как в DPO: базовые веса = reference, адаптеры = policy.

RM передаётся как объект модели в `reward_funcs`, токенизатор RM — в `reward_processing_classes`.

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

cfg = GRPOConfig(
    output_dir=f"./checkpoints/grpo/grpo-seed{random_state}",
    num_generations=4,
    generation_batch_size=4,
    max_completion_length=256,
    temperature=0.9,
    scale_rewards="group",
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    log_completions=True,
    seed=random_state,
    report_to="wandb",
    run_name=f"grpo-seed{random_state}",
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=rm_model,
    args=cfg,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    reward_processing_classes=[rm_tokenizer],
    peft_config=lora_config,
)
trainer.train()

final_dir = f"./checkpoints/grpo/grpo-seed{random_state}/final"
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"Adapters saved to {final_dir}")

## Merge + генерация

LoRA сохраняет только адаптеры. Чтобы получить полную модель:
1. Загружаем базу SFT
2. Накладываем адаптеры
3. `merge_and_unload()` — вшиваем адаптеры в веса

Результат — полный checkpoint в `merged/`, пригодный для сравнения с DPO.

In [ ]:
import os

del model, trainer
torch.cuda.empty_cache()

adapter_dir = f"./checkpoints/grpo/grpo-seed{random_state}/final"
base_model = AutoModelForCausalLM.from_pretrained(sft_path, dtype=torch.bfloat16)
base_model.resize_token_embeddings(len(tokenizer))
merged_model = PeftModel.from_pretrained(base_model, adapter_dir)
merged_model = merged_model.merge_and_unload()
merged_model = merged_model.to("cuda")

with open("eval_prompts.json") as f:
    eval_prompts = json.load(f)

merged_model.eval()
im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
eos_ids = [tokenizer.eos_token_id, im_end_id]
results = []

for i, messages in enumerate(eval_prompts[:20]):
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(merged_model.device)

    with torch.no_grad():
        outputs = merged_model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            do_sample=True,
            eos_token_id=eos_ids,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    results.append({"prompt": messages[0]["content"], "response": response})
    print(f"--- Prompt {i} ---")
    print(f"Q: {messages[0]['content'][:100]}")
    print(f"A: {response[:300]}")
    print()

out_dir = f"./checkpoints/grpo/grpo-seed{random_state}"
merged_model.save_pretrained(os.path.join(out_dir, "merged"))
tokenizer.save_pretrained(os.path.join(out_dir, "merged"))

with open(os.path.join(out_dir, "generations.json"), "w") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"Saved {len(results)} generations")
print(f"Merged model saved to {out_dir}/merged")